# PHYB57 · Chapter 11「文件读写」逐行人话版

> TXT / CSV / Pickle / JSON / HDF5 —— 五种数据文件格式，一次讲清。
> 对应课件：`PHYB57_C11.pdf`（14 页 Keynote 幻灯片）

每一段代码下面，**每一行都配了一句人话**。

## 怎么读

- 上面的代码格是**课件原文**（一字未改，连变量名和写法都照抄），可以直接运行
- 下面的「📖 这一格在做什么」是**逐行翻译**，左边标了行号，方便对着看
- 带 ⚠️ 的是**容易踩的坑**
- 带 🧪 的是**补充实验** —— 课件里没有，但能帮你亲手验证上面讲的东西
- 带 📌 的是**勘误** —— 课件说法不严谨、或与实测结果不符的地方

## 目录

| 页码 | 小节 | 主题 | 文件类型 |
|---|---|---|---|
| P1–2 | §1 | TXT 文本文件：open / write / append / read | `.txt` |
| P3 | §2 | 用 numpy 在 TXT 里存数字表格 | `.txt` |
| P4 | §3 | CSV 是什么 | `.csv` |
| P5 | §4 | 用 numpy 读写 CSV | `.csv` |
| P6 | §5 | 升级到 pandas | `.csv` |
| P7–9 | §6–7 | Pickle：序列化、兼容性、以及安全问题 | `.pkl` |
| P10–11 | §8 | JSON：跨语言的数据交换格式 | `.json` |
| P12–14 | §9 | HDF5：为大数据而生的层级格式 | `.hdf5` |
| — | §10 | 一张表收尾：五种格式怎么选 | — |

## 先记住这张「数据流地图」

课件从头到尾其实只在讲同一件事：**把内存里的 Python 对象，搬到磁盘上，再搬回来。**

| 格式 | 内存里的样子 | 磁盘上的样子 | 搬出去 | 搬回来 |
|---|---|---|---|---|
| TXT | `str` | 纯文本 | `f.write()` | `f.read()` |
| CSV | `ndarray` / `DataFrame` | 逗号分隔的文本 | `np.savetxt()` | `np.loadtxt()` |
| Pickle | 几乎任何 Python 对象 | 二进制 | `pickle.dump()` | `pickle.load()` |
| JSON | `dict` / `list` | 文本（键值对） | `json.dump()` | `json.load()` |
| HDF5 | 超大 ndarray | 二进制（带层级） | `hf[...] = arr` | `hf[...][:]` |

看懂了这张表，这一章就学会一半了。

## §0 · 开跑之前

本章要用到三个**第三方库**（不是 Python 自带的），没装的话先装：

```bash
pip install numpy pandas h5py
```

- `numpy`、`pandas` —— 科学计算的标配，多半已经有了
- `h5py` —— 只有 §9 的 HDF5 用得到，**最容易漏装的就是它**。没装的话跑到 §9 会报 `ModuleNotFoundError: No module named 'h5py'`

`json`、`pickle`、`csv` 是**标准库**，开箱即用，不用装。

In [ ]:
# 🧪 补充：把本章生成的所有数据文件关进 phyb57_c11_data/ 子文件夹，免得弄乱你的目录
import os

if os.path.basename(os.getcwd()) != 'phyb57_c11_data':
    os.makedirs('phyb57_c11_data', exist_ok=True)
    os.chdir('phyb57_c11_data')

print('数据文件会存在这里：', os.getcwd())

数据文件会存在这里： /private/tmp/phyb57/exec/phyb57_c11_data


---
# §1 · TXT 文本文件

## 课件原文（P1）

**Text Files Overview:**

- Text files contain only plain text and are often used to store data.
- Common file extension: `.txt`
- Programs expect text files to follow a specific format.

**Python `open()` Function:**

- Used to work with text files in Python by returning a *file object*.
- Syntax: `f = open(filename, mode)`

**Common modes:** `'r'` · `'w'` · `'a'` · `'r+'` · `'w+'` · `'a+'`

**Best Practice:** Always close files after use with `f.close()` to avoid data loss.

## 人话版

- **文本文件**里只有纯文字：没有颜色、没有加粗、没有表格线，记事本能打开的那种
- 扩展名通常是 `.txt`
- 程序读它的时候会**按约定的格式**去理解：几列？逗号还是空格分隔？有没有表头？所以「格式」对上了才读得对
- `open()` 是打开文件的入口。它**不返回文件内容**，而是返回一个「文件对象」—— 你可以把它想成一根**插在文件上的水管**，之后的读写都通过这根管子进行
- 用法：`f = open(文件名, 模式)`
- 用完**一定要关**，`f.close()`。不关的后果：写进去的东西可能还卡在缓冲区里没真正落盘

## 六种模式对照表（比课件更细，建议截图存下来）

| 模式 | 名字 | 能读 | 能写 | 文件不存在时 | 打开后指针在 | 会不会清空原内容 |
|---|---|---|---|---|---|---|
| `'r'` | 只读（默认） | ✅ | ❌ | ❌ 报 FileNotFoundError | 开头 | 不会 |
| `'w'` | 只写 | ❌ | ✅ | ✅ 新建 | 开头 | ⚠️ **会清空！** |
| `'a'` | 追加 | ❌ | ✅ | ✅ 新建 | 结尾 | 不会 |
| `'r+'` | 读 + 写 | ✅ | ✅ | ❌ 报 FileNotFoundError | 开头 | 不会 |
| `'w+'` | 写 + 读 | ✅ | ✅ | ✅ 新建 | 开头 | ⚠️ **会清空！** |
| `'a+'` | 追加 + 读 | ✅ | ✅ | ✅ 新建 | 结尾 | 不会 |

> ⚠️ **本章第一个大坑**：`'w'` 不是「写入」，是「**先清空、再写入**」。文件里原本有 1000 行实验数据，一句 `open('data.txt', 'w')` 就全没了 —— 而此时你连一个字都还没写进去。
>
> 记忆法：带 `w` 的都「先擦干净再写」；带 `a` 的都「接在后面写」；带 `+` 的表示又能读又能写。

## §1.1 写入文件（Writing to a File）

课件这里没有文字说明，只有一句 `Example:` 和下面这段代码 —— 它会往 `test.txt` 里写 5 行字。

In [ ]:
f = open('test.txt', 'w')
for i in range(5):
    f.write(f"This is line {i}\n")
f.close()

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `f = open('test.txt', 'w')` | 以「写」模式打开 `test.txt`，拿到文件对象 `f`。`'w'` 的含义是：**没有就新建，已经有就先清空** |
| 2 | `for i in range(5):` | 循环 5 次，`i` 依次取 0、1、2、3、4 |
| 3 | `    f.write(f"This is line {i}\n")` | 往文件里写一行字。`f"..."` 会把 `{i}` 替换成当前数字；`\n` 是换行符。⚠️ **`write()` 不会自动换行**，不写 `\n` 的话 5 行会挤成一行 |
| 4 | `f.close()` | 关闭文件。这一步是**真的在做事**：它把缓冲区里的内容刷到磁盘，并释放这个文件 |

> ⚠️ 注意这一格**没有任何输出** —— 没写 `print`，不代表没干活。想看结果，运行下面 §1.3 的读取代码。
>
> ⚠️ 还要注意 `write()` 的**缩进**：它在 `for` 循环里面（缩进 4 格），所以会执行 5 次。如果顶格写，就只写 1 次 —— 这是初学者最常见的缩进错误。

## §1.2 追加到文件（Appending to a File）

同样只有 `Example:` 和代码。注意它**接着上一个文件**写，并没有把前面 5 行冲掉。

In [ ]:
f = open('test.txt', 'a')
f.write("This is another line\n")
f.close()

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `f = open('test.txt', 'a')` | 这次用 `'a'`（append，追加）。文件已经存在，所以**不会清空**，新内容接在最后一行后面 |
| 2 | `f.write("This is another line\n")` | 写一行字。这里没有 `f"..."`，因为不需要插入变量，普通字符串就行 |
| 3 | `f.close()` | 关闭文件 |

> 💡 **`'w'` 换成 `'a'` 只差一个字母，结果天差地别**：这一格如果把 `'a'` 写成 `'w'`，前面写好的 5 行会瞬间消失，文件里只剩这一行。
>
> 想「记日志」式的持续追加，永远用 `'a'`。

## §1.3 读取文件（Reading from a File）— 一次性全读

课件列了两种读法：「Reading entire content」和「Reading line by line」。这是第一种。

In [ ]:
f = open('test.txt', 'r')
content = f.read()
f.close()

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `f = open('test.txt', 'r')` | 以「只读」模式打开。`'r'` 是默认值，其实写不写都一样 |
| 2 | `content = f.read()` | **一次性**把整个文件读成**一个长字符串**（里面的 `\n` 还保留着）。文件不大的时候最省事 |
| 3 | `f.close()` | 关闭文件 |

> ⚠️ 这一格**看起来没反应**，因为它没有 `print` —— 但内容确实已经读进 `content` 这个变量里了。这是 Jupyter 里很常见的误会：「没输出」≠「没执行」。
>
> 变量名叫 `content`（单数），因为 `read()` 返回的是一整个 `str`。

## §1.4 逐行读取（Reading line by line）

第二种读法。注意变量名只多了一个 `s`。

In [ ]:
f = open('test.txt', 'r')
contents = f.readlines()
f.close()

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `f = open('test.txt', 'r')` | 只读打开 |
| 2 | `contents = f.readlines()` | 读成**一个列表**：每一行是列表里的一个元素，行尾的 `\n` 也一起保留。适合需要逐行处理（跳过注释行、按列切分等）的场景 |
| 3 | `f.close()` | 关闭文件 |

> ⚠️ **`content` 和 `contents` 只差一个 s**，而课件正好把这两个变量名并排放在一起，特别容易看串。它们的区别是：
>
> - `content` = `str` → 一整个大字符串
> - `contents` = `list` → 一行一个元素的列表

---

🧪 下面这个格子把两者的差别并排打出来，亲眼看一次就不会记混了。

In [ ]:
# 🧪 补充实验：read() 和 readlines() 到底差在哪
print('read()      → 类型：', type(content).__name__)
print('             → 内容：', repr(content))
print()
print('readlines() → 类型：', type(contents).__name__)
print('             → 内容：', repr(contents))

read()      → 类型： str
             → 内容： 'This is line 0\nThis is line 1\nThis is line 2\nThis is line 3\nThis is line 4\nThis is another line\n'

readlines() → 类型： list
             → 内容： ['This is line 0\n', 'This is line 1\n', 'This is line 2\n', 'This is line 3\n', 'This is line 4\n', 'This is another line\n']


看到区别了吗：

- `read()` 给你**一根长绳子**（一个字符串），`\n` 只是绳子上的记号
- `readlines()` 给你**一串珠子**（一个列表），一行一颗，而且每颗珠子上都还挂着 `\n`

`repr()` 的作用是把字符串**原样显示**（连 `\n` 都画出来），而不是真的换行。调试的时候特别好用。

---

🧪 **课件说的「Best Practice」到底好在哪？** 课件只说「记得 close」，但人总会忘。Python 提供了更省心的写法。

In [ ]:
# 🧪 补充实验：更安全的 with 写法 —— 离开缩进块自动关闭文件，想忘也忘不掉
with open('test.txt', 'r') as f:
    for line in f:                 # 直接遍历文件对象 = 一行一行读，不用先全读进内存
        print(line.rstrip())       # rstrip() 去掉行尾的换行符，不然每行后面会多一个空行

This is line 0
This is line 1
This is line 2
This is line 3
This is line 4
This is another line


三行代码，把课件前面所有写法的毛病一次解决：

| 写法 | 会不会自动关闭 | 会不会一次吃进内存 | 推荐度 |
|---|---|---|---|
| `f = open(...)` + `f.close()` | ❌ 全靠你记性 | 取决于用哪个读法 | 能用，但容易忘 |
| `with open(...) as f:` | ✅ 自动关，**报错也会关** | 逐行读就不吃内存 | ✅ **推荐** |

> 💡 为什么 `with` 更强？因为它连「程序中途抛异常」的情况都照顾到了 —— 无论怎么退出，`with` 都保证文件被关上。而手写的 `f.close()` 一旦前面的代码报错，就永远执行不到了。
>
> 实际工作中**几乎没人手写 `close()`**，一律用 `with`。

---
# §2 · 用 numpy 在 TXT 里存数字表格

## 课件原文（P3）

**Working with Numbers and Arrays in TXT Files**

**Saving Numerical Data:**

- Using numpy to handle arrays:
- Saves a 2D array to `my_arr.txt` with a custom header.

**Loading Numerical Data:**

- Reading arrays back into Python:
- Automatically skips the header and loads the array back into memory.

## 人话版

上一节用 `write()` 一行一行写字符串，太累了。要存**一整张数字表格**，用 numpy 一行就够：

- `np.savetxt()` = save txt，把数组**存**成纯文本
- `np.loadtxt()` = load txt，把纯文本**读**回数组
- 两个函数名字对称，记住一个就记住另一个

In [ ]:
import numpy as np
arr = np.array([[1.20, 2.20, 3.00], [4.14, 5.65, 6.42]])
np.savetxt('my_arr.txt', arr, fmt='%.2f', header='Col1 Col2 Col3')

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `import numpy as np` | 导入 numpy，起个通用简称 `np`（全世界都这么写） |
| 2 | `arr = np.array([[1.20, 2.20, 3.00], [4.14, 5.65, 6.42]])` | 建一个 **2 行 3 列的二维数组**（就是一张小表格）。外面那对 `[]` 是「行」，里面每对 `[]` 是一行 |
| 3 | `np.savetxt('my_arr.txt', arr, fmt='%.2f', header='Col1 Col2 Col3')` | 把数组存成文本文件。参数拆开看：`'my_arr.txt'` 存到哪，`arr` 存什么，`fmt='%.2f'` 每个数保留 **2 位小数**，`header=...` 在文件开头加一行表头 |

---

🧪 「`fmt='%.2f'`」听起来抽象，直接看文件内容最快：

In [ ]:
# 🧪 补充实验：亲眼看一看 savetxt 到底写进去了什么
print(open('my_arr.txt').read())

# Col1 Col2 Col3
1.20 2.20 3.00
4.14 5.65 6.42



注意第一行 —— 我们写的是 `'Col1 Col2 Col3'`，但文件里变成了：

```text
# Col1 Col2 Col3
```

> ⚠️ **`savetxt` 会自动在 header 前面补上 `# `**（井号 + 空格）。这不是 bug，恰恰是设计：
>
> - 「`#`」在文本文件里是**注释**的意思
> - 下一格的 `loadtxt` 正是靠这个 `#` 才知道「这行是表头、不是数据」，于是自动跳过
> - 两个函数靠这个约定**默契配合**，你什么都不用管
>
> 如果哪天你自己手写文件、忘了写 `#`，`loadtxt` 就会把这行表头当成数据去解析，然后报错。

In [ ]:
my_arr = np.loadtxt('my_arr.txt')

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `my_arr = np.loadtxt('my_arr.txt')` | 把刚才那个文件读回成 numpy 数组。默认按**空白**（空格 / Tab）切分，并且**自动跳过以 `#` 开头的行**（也就是表头） |

> ⚠️ 两个提醒：
>
> 1. 这一格同样**没有 print**，所以画面上什么都不会出现 —— 数据已经回到 `my_arr` 里了
> 2. 变量名 `my_arr` **和第 2 行那个原数组重名**，所以原来的 `arr` 还在，而 `my_arr` 已经被换成从文件读回来的版本了

🧪 眼见为实：

In [ ]:
# 🧪 补充实验：读回来的到底是不是原样？
print(my_arr)
print()
print('形状：', my_arr.shape, ' 类型：', my_arr.dtype)
print('和原数组一模一样吗？', np.allclose(arr, my_arr))

[[1.2  2.2  3.  ]
 [4.14 5.65 6.42]]

形状： (2, 3)  类型： float64
和原数组一模一样吗？ True


`np.allclose()` 是浮点数的比较方式，返回 `True` 说明：**存下去再读回来，数字一个都没丢**。

> 💡 为什么不用 `==` 比较？因为浮点数在计算机里是近似存储的，`0.1 + 0.2 != 0.3` 这种事真的会发生。判断两个浮点数组是否相等，要用 `np.allclose()`（允许极小的误差）。这个习惯在物理数据处理里非常重要。
>
> ⚠️ 另外注意 `my_arr` 的形状是 `(2, 3)` —— `loadtxt` 读回来**还是二维的**。但如果文件里只有一列，读回来会变成一维数组（形状是 `(2,)` 而不是 `(2, 1)`），这是很多人第一次踩的坑。

---
# §3 · CSV 文件是什么

## 课件原文（P4）

**CSV Files in Python**

**What are CSV Files?**

- CSV stands for **Comma-Separated Values**, a simple and widely used file format for storing tabular data.
- Each line in a CSV file represents a data record, and each record consists of one or more fields separated by commas.
- CSV files are highly compatible with applications like Microsoft Excel, making them easy to visualize.

**Python's CSV Handling:**

- Python provides several tools to handle CSV files, including the `csv` module and third-party libraries like numpy and pandas.
- In scientific applications, CSV files often store large tables of numerical data, which can be efficiently managed using Python.

## 人话版

CSV 就是**用逗号当分隔符的纯文本表格**。它没有任何花哨的格式，但正因为如此，什么软件都能打开它。

```
name,energy,valid
p1,12.4,True
p2,8.1,False
```

- 第一行通常是**表头**（列名）
- 之后每一行是**一条记录**
- 记录里的每个值用**逗号**隔开
- 因为 Excel 双击就能打开，所以它是「给人类看的数据库」

### TXT vs CSV：差在哪？

| | TXT | CSV |
|---|---|---|
| 分隔符 | 随便（空格、Tab 都行） | **逗号**（约定俗成） |
| 有没有表头 | 看心情（numpy 用 `#` 标记） | 通常第一行就是表头 |
| Excel 能直接打开吗 | 要手动导入、选分隔符 | ✅ 双击就行 |
| 适合什么 | 日志、说明、自由文本 | **表格数据**、实验记录 |

> 💡 一句话：**CSV 是 TXT 的一个「约定」版本** —— 约定用逗号分隔、约定第一行是列名。正是这个约定，让 Excel / pandas / MATLAB 都能无脑读懂它。

---

🧪 手写一个最小 CSV，看看它到底长什么样（顺便用 PHYB57 熟悉的粒子探测器数据）：

In [ ]:
# 🧪 补充实验：手写一个最小 CSV
with open('mini.csv', 'w') as f:
    f.write('name,energy,valid\n')
    f.write('p1,12.4,True\n')
    f.write('p2,8.1,False\n')

print(open('mini.csv').read())

name,energy,valid
p1,12.4,True
p2,8.1,False



### 课件提到但没给代码：标准库 `csv` 模块

课件原文说「Python provides several tools... including the `csv` module」，但后面只演示了 numpy 和 pandas。这里补上 `csv` 模块的用法（它是**标准库**，不用装）。

In [ ]:
# 🧪 补充实验：标准库 csv 模块 —— 不装 numpy / pandas 也能读写 CSV
import csv

with open('mini.csv', newline='') as f:
    for row in csv.reader(f):
        print(row)

['name', 'energy', 'valid']
['p1', '12.4', 'True']
['p2', '8.1', 'False']


⚠️ 注意输出的每一行都是一个**列表**，而且里面全是**字符串**：`'12.4'` 而不是 `12.4`。

> 💡 这是读文本文件的铁律：**从文件里读出来的一切都是字符串**。要用数字得自己转：`float(row[1])`。文件本身不记录「这一列是数字」，它只是一堆字符。
>
> 另外 `newline=''` 是 `csv` 模块官方要求的写法，不写的话在 Windows 上会出现空行。

`DictReader` 更贴心，直接把每行变成字典（用表头当键）：

In [ ]:
# 🧪 补充实验：DictReader —— 用列名取值，不用数第几列
with open('mini.csv', newline='') as f:
    for row in csv.DictReader(f):
        print(row, '→ energy =', row['energy'])

{'name': 'p1', 'energy': '12.4', 'valid': 'True'} → energy = 12.4
{'name': 'p2', 'energy': '8.1', 'valid': 'False'} → energy = 8.1


> 💡 `DictReader` 的好处：以后列的顺序变了，`row['energy']` 照样能用；而用 `row[1]` 的写法就得跟着改。**用列名比用序号更抗变化**。

---

# §4 · 用 numpy 读写 CSV

## 课件原文（P5）

**Writing and Reading CSV Files Using Numpy**

**Writing a CSV File:**

- Python's numpy package can be used to write CSV files efficiently.
- Example code to generate and save random data to a CSV file:
- The code creates a file with 100 rows and 5 columns of random numbers, saved in CSV format.

**Reading a CSV File:**

- The same numpy package allows reading CSV files into arrays for easy data manipulation:
- This reads the CSV back into a numpy array and prints the first five rows.

## 人话版

有了 §2 的基础，这一节其实只多了一个参数：`delimiter=','`。

- 存的时候告诉 numpy「用逗号分隔」
- 读的时候再告诉它一遍「用逗号分隔」
- 两边必须**说好同一件事**，否则读回来就是一锅粥

In [ ]:
import numpy as np
data = np.random.random((100, 5))
np.savetxt('test.csv', data, fmt='%.2f', delimiter=',', header='c1, c2, c3, c4, c5')

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `import numpy as np` | 导入 numpy |
| 2 | `data = np.random.random((100, 5))` | 造一个 **100 行 5 列**的随机数矩阵，每个数在 0 到 1 之间。`(100, 5)` 是**形状**：100 行、5 列（先写行、后写列） |
| 3 | `np.savetxt('test.csv', data, fmt='%.2f', delimiter=',', header='c1, c2, c3, c4, c5')` | 存成 CSV。跟 §2 的唯一区别就是多了 `delimiter=','` —— **用逗号分隔**（不写的话默认用空格，那就不是 CSV 了）。表头字符串里的逗号也要自己写 |

> ⚠️ 细节：`header='c1, c2, c3, c4, c5'` 里逗号后面带了**空格**，所以存进文件的表头是 `# c1, c2, c3, c4, c5`。这些空格后面会让 pandas 的列名带上前导空格 —— §5 会亲眼看到这个坑。
>
> 💡 `np.random.random()` 每次运行结果都不一样（真随机）。所以你的输出数字和这里的不一样，是**正常的**。

---

🧪 看看前 3 行长什么样：

In [ ]:
# 🧪 补充实验：看清 CSV 文件的真实模样（只取前 3 行，不然 100 行会刷屏）
with open('test.csv') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.rstrip())
print('...')

# c1, c2, c3, c4, c5
0.16,0.89,0.54,0.80,0.66
0.56,0.79,0.04,0.73,0.31
...


看清楚了：`# c1, c2, c3, c4, c5` 是表头，下面每行 5 个数、用逗号隔开、保留 2 位小数 —— 完全就是 `fmt` 和 `delimiter` 说了算。

In [ ]:
my_csv = np.loadtxt('test.csv', delimiter=',')
print(my_csv[:5, :])

[[0.16 0.89 0.54 0.8  0.66]
 [0.56 0.79 0.04 0.73 0.31]
 [0.8  0.93 0.95 0.04 0.62]
 [0.75 0.56 0.02 0.84 0.21]
 [0.8  0.84 0.36 0.54 0.26]]


**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `my_csv = np.loadtxt('test.csv', delimiter=',')` | 读回来。**必须再告诉它一遍分隔符是逗号**。表头那行以 `#` 开头，被自动跳过 |
| 2 | `print(my_csv[:5, :])` | 打印**前 5 行、所有列**。`[:5, :]` 是 numpy 的切片语法：**逗号前管行、逗号后管列**，`:` 表示「全都要」 |

> 💡 为什么只打印 5 行？因为 100 行全打出来会刷屏，看不清。`[:5, :]` 就是「只看前 5 行」的意思。

🧪 把这个切片语法彻底搞明白（这是 numpy 最核心的语法之一）：

In [ ]:
# 🧪 补充实验：拆解 my_csv[:5, :] —— 逗号左边是行，右边是列
print('my_csv.shape    →', my_csv.shape)          # 整个数组：100 行 5 列
print()
print('my_csv[:5, :]   → 前 5 行、所有列   shape =', my_csv[:5, :].shape)
print('my_csv[:, 0]    → 所有行、第 0 列   shape =', my_csv[:, 0].shape)
print('my_csv[0, :]    → 第 0 行、所有列   shape =', my_csv[0, :].shape)
print('my_csv[2, 3]    → 第 2 行第 3 列那一个数 =', my_csv[2, 3])

my_csv.shape    → (100, 5)

my_csv[:5, :]   → 前 5 行、所有列   shape = (5, 5)
my_csv[:, 0]    → 所有行、第 0 列   shape = (100,)
my_csv[0, :]    → 第 0 行、所有列   shape = (5,)
my_csv[2, 3]    → 第 2 行第 3 列那一个数 = 0.04


记住这个「**行, 列**」的写法，之后处理任何实验数据都靠它。

| 写法 | 含义 | 结果形状 |
|---|---|---|
| `a[:5, :]` | 前 5 行，所有列 | `(5, 5)` |
| `a[:, 0]` | 所有行，第 0 列 | `(100,)` ← 注意掉了一维 |
| `a[0, :]` | 第 0 行，所有列 | `(5,)` |
| `a[2, 3]` | 单个元素 | 一个标量 |

---

🧪 **如果忘了写 `delimiter` 会怎样？** 这是 CSV 最常见的错误，亲眼看一次报错，以后就知道怎么修了：

In [ ]:
# 🧪 补充实验：故意犯错 —— 读 CSV 却忘了写 delimiter
try:
    np.loadtxt('test.csv')
except ValueError as e:
    print('报错了 →', e)

报错了 → could not convert string '0.16,0.89,0.54,0.80,0.66' to float64 at row 0, column 1.


报错信息说得很清楚：**「could not convert string '...' to float64」** —— numpy 把整行 `0.46,0.13,0.82,0.56,0.89` 当成了**一个**字符串，然后试图把它变成数字，当然失败。

> 💡 以后看到这个 `could not convert string ... to float64` 的报错，第一反应就应该是：**分隔符没写对**（或者文件里有非数字的杂质行）。
>
> 修法：`np.loadtxt('test.csv', delimiter=',')` 补上逗号即可。

---

# §5 · CSV 进阶：换用 pandas

## 课件原文（P6）

**Beyond Numpy for CSV Handling**

**Advanced CSV Handling with Pandas:**

- While numpy works well for numerical data, pandas provides more advanced capabilities, especially for working with labeled data and DataFrames.
- pandas supports easier manipulation of CSV data, including data cleaning, filtering, and summarization.
- Example code to use pandas for CSV operations:
- This provides a richer interface for working with CSV files, especially for tabular data analysis.

## 人话版

numpy 把表格当成**纯数字矩阵** —— 它只认「第几行第几列」，不认列名。

pandas 把表格当成**带标签的表格**（DataFrame）—— 列有名字，行有编号，还能直接 `df['c2'].mean()`。

| | numpy | pandas |
|---|---|---|
| 数据模型 | `ndarray`（纯数字方块） | `DataFrame`（带行名列名的表格） |
| 取一列 | `a[:, 1]`（要记住是第 1 列） | `df['c2']`（用名字） |
| 读 CSV | `np.loadtxt`（**必须**给 delimiter） | `pd.read_csv`（**自动**猜逗号） |
| 适合 | 纯数值计算、矩阵运算 | 数据分析、清洗、统计、混合类型 |

> 💡 一句话：**做矩阵运算用 numpy，做数据分析用 pandas**。两者不是替代关系，pandas 底层就是 numpy。

In [ ]:
import pandas as pd
df = pd.read_csv('test.csv')
print(df.head())

   # c1    c2    c3    c4    c5
0  0.16  0.89  0.54  0.80  0.66
1  0.56  0.79  0.04  0.73  0.31
2  0.80  0.93  0.95  0.04  0.62
3  0.75  0.56  0.02  0.84  0.21
4  0.80  0.84  0.36  0.54  0.26


**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `import pandas as pd` | 导入 pandas，通用简称 `pd` |
| 2 | `df = pd.read_csv('test.csv')` | 一行把 CSV 读成 DataFrame。**不用自己写分隔符**，pandas 默认就按逗号切（它还会自动推断每列的类型） |
| 3 | `print(df.head())` | `head()` 默认显示**前 5 行**。注意左边多了一列 0、1、2、3、4 —— 那是 pandas 自动加的**行索引** |

> ⚠️ **注意列名！** 输出里的列名是 `# c1`、` c2`、` c3`…… 而不是干净的 `c1`、`c2`。原因有两个：
>
> 1. `np.savetxt` 自动给表头加了 `# ` 前缀，而 **pandas 不会像 `loadtxt` 那样跳过 `#` 行** —— 它老老实实把 `# c1` 当成了第一个列名
> 2. 我们写表头时逗号后带了空格，所以 `c2` 变成了 ` c2`（前面多一个空格）
>
> 这是 **numpy 和 pandas 之间一个很真实的差异**：`loadtxt` 认 `#` 是注释，`read_csv` 不认。课件把两个例子并排放在一起，但没有点出这个区别。

🧪 把列名打出来看一眼：

In [ ]:
# 🧪 补充实验：pandas 眼里的列名到底是什么？
print('列名：', list(df.columns))
print()
print("想取第一列，得写成 df['# c1'] —— 是不是很难看？")

列名： ['# c1', ' c2', ' c3', ' c4', ' c5']

想取第一列，得写成 df['# c1'] —— 是不是很难看？


🧪 那就自己指定干净的列名（`names=` 给新列名，`header=0` 表示「原来的第一行是表头，跳过它」）：

In [ ]:
# 🧪 补充实验：自己指定列名，世界清爽了
df2 = pd.read_csv('test.csv', names=['c1', 'c2', 'c3', 'c4', 'c5'], header=0)
print(df2.head())
print()
print('列名：', list(df2.columns))

     c1    c2    c3    c4    c5
0  0.16  0.89  0.54  0.80  0.66
1  0.56  0.79  0.04  0.73  0.31
2  0.80  0.93  0.95  0.04  0.62
3  0.75  0.56  0.02  0.84  0.21
4  0.80  0.84  0.36  0.54  0.26

列名： ['c1', 'c2', 'c3', 'c4', 'c5']


🧪 顺便见识一下 pandas 真正的强项 —— **一行拿到统计摘要**：

In [ ]:
# 🧪 补充实验：pandas 的杀手锏
print('形状（行, 列）：', df2.shape)
print()
print("按列名取一列并算平均值：df2['c2'].mean() =", df2['c2'].mean())
print()
print('一行拿到全部统计量：')
print(df2.describe())

形状（行, 列）： (100, 5)

按列名取一列并算平均值：df2['c2'].mean() = 0.4788

一行拿到全部统计量：
               c1          c2          c3          c4          c5
count  100.000000  100.000000  100.000000  100.000000  100.000000
mean     0.484900    0.478800    0.446700    0.459800    0.481800
std      0.286467    0.282318    0.302672    0.301132    0.272343
min      0.020000    0.010000    0.020000    0.010000    0.020000
25%      0.220000    0.237500    0.202500    0.150000    0.260000
50%      0.485000    0.470000    0.435000    0.445000    0.470000
75%      0.692500    0.750000    0.680000    0.732500    0.720000
max      0.990000    0.970000    1.000000    0.970000    0.990000


`describe()` 一口气给出了每列的：**个数、均值、标准差、最小值、四分位数、最大值**。

> 💡 在实验课里，拿到一组新数据后第一件事往往就是 `df.describe()` —— 三秒钟就能发现「这列怎么有个 -999」「这列的均值明显不对」之类的异常。这是 numpy 要写好几行才能做到的事。
>
> | 你想做的事 | numpy | pandas |
> |---|---|---|
> | 按列名取一列 | `a[:, 2]` | `df['c3']` |
> | 求一列均值 | `a[:, 2].mean()` | `df['c3'].mean()` |
> | 全部统计量 | 自己一个个算 | `df.describe()` |
> | 按条件筛选行 | `a[a[:, 2] > 0.5]` | `df[df['c3'] > 0.5]` |

---
# §6 · Pickle：把 Python 对象原样冻起来

## 课件原文（P7）

**Pickle Files**

**Overview:**

- Pickle is a Python module used for **serializing** and **de-serializing** Python object structures.
- Serialization involves converting objects in memory (e.g., dictionaries, lists) into a **byte stream** for storage on disk as a binary file.
- De-serialization is the reverse process, where the byte stream is converted back into its original object form.

**When to Use:**

- Useful for saving Python-specific data types, like dictionaries, lists, or tuples, for later use.
- Ideal for sharing data with other Python programs but has limitations when sharing across different programming languages.

**Key Functions:**

- `pickle.dump()`: Serializes an object and writes it to a file.
- `pickle.load()`: Reads a pickle file and de-serializes it back into the original Python object.

## 人话版

前面几节存的都是**数字**。但如果我想存一个**字典**、一个**自定义对象**、甚至一个训练好的模型呢？CSV 存不下这些结构。

Pickle 就是干这个的：**把内存里的 Python 对象整个「冻」成二进制，之后原样「解冻」回来**。

- **序列化**（serialize）= 冻起来：内存对象 → 字节流
- **反序列化**（deserialize）= 解冻：字节流 → 内存对象

> 🧊 打个比方：CSV 像是把你的数据**抄写在纸上**（别人也能读，但复杂的结构抄不下来）；Pickle 像是把东西**整个真空封存**（形状、类型原封不动，但只有 Python 能拆封）。

### 四个函数，别搞混

| 函数 | 作用 | 去处 |
|---|---|---|
| `pickle.dump(obj, f)` | 序列化**到文件** | 磁盘 |
| `pickle.load(f)` | 从**文件**反序列化 | 内存 |
| `pickle.dumps(obj)` | 序列化成 `bytes` | 内存变量 |
| `pickle.loads(b)` | 从 `bytes` 反序列化 | 内存 |

> 💡 记忆法：**带 `s` 的（dumps / loads）处理字符串/字节，不碰文件**。`s` = string。这个规律对 `json` 模块**完全一样**（`json.dump` / `json.dumps`）。

## §6.1 写一个 Pickle 文件（Pickling）

课件 P8 的正文写了一句「First, import the pickle module: `import pickle`」，但**代码截图里没有画出这一行** —— 所以先把它补上，不然下一格会直接报 `NameError`。

In [ ]:
# 🧪 补充：课件正文要求「先 import pickle」，但代码截图漏画了这一行，补上
import pickle

In [ ]:
dict_a = {'A':0, 'B':1, 'C':2}
pickle.dump(dict_a, open('test.pkl', 'wb'))

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `dict_a = {'A':0, 'B':1, 'C':2}` | 一个字典：键是 `'A'`/`'B'`/`'C'`，值是 0/1/2 |
| 2 | `pickle.dump(dict_a, open('test.pkl', 'wb'))` | 把字典序列化后写进 `test.pkl`。`'wb'` = **w**rite **b**inary（二进制写）—— pickle 产出的是**字节**不是文字，所以模式里**必须带 `b`**，写成 `'w'` 会报 `TypeError` |

> ⚠️ **两个坑，都在这一行里：**
>
> 1. `open(...)` 被嵌在函数调用里，**这个文件对象没有被 close**。课件这么写能跑（靠 CPython 的引用计数垃圾回收兜底），但**不是好习惯**，换个 Python 实现就可能出问题。
> 2. 更隐蔽的：`pickle.dump` 只是把数据交给**写缓冲区**，缓冲区什么时候真正落盘由 Python 决定。如果程序在这之前异常退出，文件可能就是**残缺的**。
>
> 结论：课件这么写是为了截图短，你自己写请用下面 `with` 的写法。

🧪 规范写法对比：

In [ ]:
# 🧪 补充实验：规范写法 —— 用 with 保证文件一定被关闭、数据一定落盘
with open('test2.pkl', 'wb') as f:
    pickle.dump(dict_a, f)

with open('test2.pkl', 'rb') as f:
    print('读回来：', pickle.load(f))
print('两种写法存的文件一样大吗？', __import__('os').path.getsize('test.pkl') == __import__('os').path.getsize('test2.pkl'))

读回来： {'A': 0, 'B': 1, 'C': 2}
两种写法存的文件一样大吗？ True


🧪 **Pickle 真正的价值：它能装下 JSON 装不了的东西。**

课件说 Pickle「useful for saving Python-specific data types」，但没演示到底有多能装。看一眼：

In [ ]:
# 🧪 补充实验：pickle 什么都能装 —— 自定义对象、元组、numpy 数组，一锅端
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

obj = {
    'array': np.array([1.5, 2.5, 3.5]),
    'student': Student('Alice', 92),
    'tuple': (1, 2),          # 元组
    'none': None,
}

with open('mixed.pkl', 'wb') as f:
    pickle.dump(obj, f)

with open('mixed.pkl', 'rb') as f:
    back = pickle.load(f)

print('数组   ：', back['array'], '  类型还是', type(back['array']).__name__)
print('对象   ：', back['student'].name, back['student'].score)
print('元组   ：', back['tuple'], '  类型还是', type(back['tuple']).__name__)
print()
print('💡 连你自己定义的对象都能原样还原回来 —— CSV / JSON 都做不到这件事。')

数组   ： [1.5 2.5 3.5]   类型还是 ndarray
对象   ： Alice 92
元组   ： (1, 2)   类型还是 tuple

💡 连你自己定义的对象都能原样还原回来 —— CSV / JSON 都做不到这件事。


🔍 顺便看一眼 pickle 文件长什么样 —— 你会发现它**不是文字**：

In [ ]:
# 🧪 补充实验：pickle 文件用文本方式打开是乱码（这正是「二进制格式」的意思）
with open('test.pkl', 'rb') as f:
    raw = f.read()
print('文件里的前 60 个字节：', raw[:60])
print()
print('能用 UTF-8 解码成文字吗？', end=' ')
try:
    raw.decode('utf-8')
    print('能（那它其实是文本）')
except UnicodeDecodeError:
    print('❌ 不能 —— 这就是为什么叫「二进制格式」')

文件里的前 60 个字节： b'\x80\x04\x95\x17\x00\x00\x00\x00\x00\x00\x00}\x94(\x8c\x01A\x94K\x00\x8c\x01B\x94K\x01\x8c\x01C\x94K\x02u.'

能用 UTF-8 解码成文字吗？ ❌ 不能 —— 这就是为什么叫「二进制格式」


## §6.2 读一个 Pickle 文件（Unpickling）

In [ ]:
my_dict = pickle.load(open('./test.pkl', 'rb'))

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `my_dict = pickle.load(open('./test.pkl', 'rb'))` | 先以「二进制读」打开，再把字节流**反序列化**回 Python 对象，赋值给 `my_dict`。`'rb'` = **r**ead **b**inary |

> 💡 `'./'` 表示「当前目录」。`'./test.pkl'` 和 `'test.pkl'` 完全等价，写不写随你。
>
> ⚠️ 同样**没有 close**。课件风格如此，你自己写请用 `with`。

🧪 验明正身 —— 解冻出来的，到底是不是原来那个字典？

In [ ]:
# 🧪 补充实验：解冻出来的东西，和原来一模一样吗？
print('内容    ：', my_dict)
print('类型    ：', type(my_dict).__name__)
print()
print('和原字典相等吗？  my_dict == dict_a →', my_dict == dict_a)
print('是同一个对象吗？  my_dict is dict_a →', my_dict is dict_a)

内容    ： {'A': 0, 'B': 1, 'C': 2}
类型    ： dict

和原字典相等吗？  my_dict == dict_a → True
是同一个对象吗？  my_dict is dict_a → False


`==` 是 `True`，`is` 是 `False` —— 这两个结果一起看很有意思：

> 💡 **`==` 问的是「内容一样吗」，`is` 问的是「是不是同一个东西」。**
>
> `pickle.load` 是**照原件重新造了一个复制品**：内容分毫不差（所以 `==` 为真），但它在内存里是**另一块地方**（所以 `is` 为假）。改了 `my_dict` 不会影响 `dict_a`。

---

# §7 · Pickle 的局限、兼容性 —— 以及一个课件没说的安全问题

## 课件原文（P9）

**Limitations and Compatibility**

**Compatibility Issues:**

- **Not a universal format:** While TXT and CSV files can be used across languages (e.g., R, MATLAB, Java), Pickle is specific to Python.
- **Python Version Differences:** Pickle files created in Python 2 may not be compatible with Python 3. Use `encoding='latin1'` in the `pickle.load()` function for compatibility with Python 2 files.

**Example:**

- Reading a Python 2 Pickle file in Python 3:

**Key Warning:**

- Avoid using Pickle files if you need cross-language compatibility or long-term data preservation, as formats may change across Python versions.

## 人话版

Pickle 很好用，但它是**有代价的**：

| 局限 | 意思 |
|---|---|
| 🚫 **不是通用格式** | TXT / CSV 连 R、MATLAB、Java 都能读，Pickle **只有 Python 认识** |
| 🚫 **跨 Python 版本有风险** | Python 2 存的 pickle，Python 3 可能读不了（要靠 `encoding='latin1'` 救） |
| 🚫 **不适合长期保存** | 格式会随 Python 版本变。存 5 年的实验数据，别用 pickle |
| 🚫 **不安全** | ⬅️ **课件没提，但这是最重要的一条**，见下面 |

> 💡 **记住这句话就够了**：Pickle 适合**临时缓存**（程序今天存、明天读），不适合**归档**（要存好几年、或要给别的语言用）。

In [ ]:
# 🧪 补充：课件里的 filename 是个占位符，代表「你的文件名」。这里把它指向刚才生成的 pickle 文件
filename = 'test.pkl'

In [ ]:
new_dict = pickle.load(open(filename, 'rb'), encoding='latin1')

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `new_dict = pickle.load(open(filename, 'rb'), encoding='latin1')` | 从 `filename` 读出并还原。多出来的 `encoding='latin1'` 是给 **Python 2 时代存的老 pickle** 用的：Python 2 的字符串本质是字节，用 Python 3 默认的 UTF-8 去解码会报 `UnicodeDecodeError`，指定 latin1 就能顺利读进来 |

> ⚠️ **直接照抄这一行会报错**：`NameError: name 'filename' is not defined`。它不是内置变量，是课件里的**占位符**，代表「你自己填文件名」。上面那个 🧪 格子就是替你把 `filename` 定义好。

🧪 试试看 —— 加了这个参数，Python 3 自己存的 pickle 也照样读得出来：

In [ ]:
# 🧪 补充实验：encoding='latin1' 到底有没有副作用？
new_dict = pickle.load(open('test.pkl', 'rb'), encoding='latin1')
print('读出来：', new_dict)
print()
print('💡 encoding 只决定「字节怎么变回字符串」。Python 3 自产的 pickle 本来就是 UTF-8，')
print('   所以加不加 latin1 结果都一样 —— 这个参数只在读 Python 2 老文件时才真正起作用。')

读出来： {'A': 0, 'B': 1, 'C': 2}

💡 encoding 只决定「字节怎么变回字符串」。Python 3 自产的 pickle 本来就是 UTF-8，
   所以加不加 latin1 结果都一样 —— 这个参数只在读 Python 2 老文件时才真正起作用。


## 🔒 课件没说的安全问题：Pickle 会执行代码

这是 Pickle 和 JSON **最本质的区别**，也是实际工作中最容易出事的地方：

> ⚠️ **`pickle.load()` 会执行文件里记录的指令。**
>
> Pickle 的格式里不只是「数据」，还包含「**怎么重建这个对象**」的操作指令。还原一个对象时，Python 会真的去**调用**那些指令。所以一个恶意构造的 `.pkl` 文件，可以在你 `load` 的**那一瞬间**执行任意代码 —— 删你的文件、偷偷发网络请求、装后门，都做得到。
>
> **所以：永远不要 `pickle.load` 来路不明的文件**（尤其是从网上、从别人那里拿到的 `.pkl`）。
>
> JSON 完全没有这个问题 —— 它只会**解析数据**，格式里根本没有「执行动作」这一说。

| | 会不会执行代码 | 能不能读陌生文件 |
|---|---|---|
| `pickle.load` | ⚠️ **会** | ❌ 绝对不要 |
| `json.load` | ✅ 只解析数据 | ✅ 安全 |
| `np.loadtxt` / `pd.read_csv` | ✅ 只解析数据 | ✅ 安全 |

> 💡 这也是为什么「配置文件」「API 返回的数据」全都用 JSON 而不是 Pickle —— **不只是因为跨语言，更因为安全**。

---

**这一节的小结**

| 什么时候用 Pickle | 什么时候别用 Pickle |
|---|---|
| ✅ 程序内部的临时缓存（今天存、明天读） | ❌ 要和别的语言交换数据 |
| ✅ 要保存复杂的 Python 对象（自定义类、嵌套结构） | ❌ 要长期归档（几年后可能读不了） |
| ✅ 自己的程序、自己的文件 | ❌ 来路不明的文件（**安全风险**） |

---
# §8 · JSON：跨语言的数据交换格式

## 课件原文（P10）

**JSON Files in Python**

**What is JSON?**

- JSON stands for **JavaScript Object Notation**.
- A lightweight, **language-independent** data format often used for data exchange.
- JSON files typically end with the extension `.json`.

**Key Characteristics**

- Stores data in **key-value pairs**, similar to Python dictionaries.
- Supports various data types: strings, numbers, lists, and nested objects.
- Compact and faster than Pickle for serialization (Python object to byte stream).

**Why Use JSON?**

- **Language agnostic:** Can be read by many programming languages.
- More efficient disk usage compared to Pickle.
- JSON is ideal for cross-platform data exchange, making it widely adopted in **APIs**.

## 人话版

JSON 长得很像 Python 的字典，只是**键必须用双引号**：

```json
{
  "school": "UC Berkeley",
  "address": { "city": "Berkeley", "state": "California" },
  "list": ["student 1", "student 2", "student 3"]
}
```

它和 Python 字典的对应关系几乎是 1:1：

| JSON | Python |
|---|---|
| `{ }` 对象 | `dict` |
| `[ ]` 数组 | `list` |
| `"文字"` | `str` |
| `123` / `1.5` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

### JSON vs Pickle：一张表看清

| | JSON | Pickle |
|---|---|---|
| 格式 | **纯文本**，记事本能读 | 二进制，打开是乱码 |
| 跨语言 | ✅ 几乎每种语言都能读 | ❌ 只有 Python |
| 能存什么 | 只有基本类型（字典/列表/字符串/数字/布尔/None） | **几乎任何** Python 对象 |
| 安全 | ✅ 只解析数据 | ⚠️ **会执行代码** |
| 典型用途 | 配置文件、API、跨语言交换 | 程序内部临时缓存 |

> 📌 **勘误**：课件 P10 写了「Compact and faster than Pickle」和「More efficient disk usage compared to Pickle」。
>
> 这两句**在实测中不成立**。同一份数字数据，JSON 的文件通常比 Pickle **大 2 倍以上**、写入也**慢几十倍**（§10 会给出实测数据）。
>
> JSON 真正的优势是**可读、跨语言、安全** —— 不是「更小更快」。课件这一点说得不准确，别记错了。
>
> （只有在数据量很小、且内容本来就是短字符串时，两者体积才接近。）

## §8.1 写入 JSON 文件

课件原文说「Use Python's `json` library to easily handle JSON」，注意这里的 `json` 是**标准库**，不需要 pip 安装。

In [ ]:
import json
school = {
    "school": "UC Berkeley",
    "address": {
        "city": "Berkeley",
        "state": "California",
        "postal": "94720"
    },
    "list": ["student 1", "student 2", "student 3"]
}
json.dump(school, open('school.json', 'w'))

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `import json` | 导入 json 标准库 |
| 2–10 | `school = { ... }` | 定义一个**嵌套字典**：最外层有 3 个键。`"address"` 的值又是**一个字典**（字典套字典），`"list"` 的值是**一个列表**（字典套列表） |
| 11 | `json.dump(school, open('school.json', 'w'), )` | 把字典写成 JSON 文件。`'w'` 是**文本**写（不是二进制）—— 因为 JSON 本质是文字 |

> 💡 注意第 3–7 行的**缩进**。缩进不是给 Python 看的（字典里不需要），是给**人**看的 —— 这样一眼就能看出 `address` 是嵌套在里面的。
>
> ⚠️ 但 Python 的缩进**不会**被写进文件！`json.dump` 默认会把所有内容**挤成一行**。不信往下看。

🧪 看看生成的 JSON 到底长什么样：

In [ ]:
# 🧪 补充实验：默认输出是挤成一行的
print(open('school.json').read())

{"school": "UC Berkeley", "address": {"city": "Berkeley", "state": "California", "postal": "94720"}, "list": ["student 1", "student 2", "student 3"]}


确实挤成了一行 —— **虽然 Python 里的缩进很漂亮，但文件里一点都没保留**。

🧪 想要人能读的 JSON，加两个参数（这是实际工作中**最常用的两个参数**）：

In [ ]:
# 🧪 补充实验：indent=2 缩进美化 + ensure_ascii=False 保留中文
data_cn = {"课程": "PHYB57", "章节": 11, "标签": ["文件", "IO"]}
json.dump(data_cn, open('school_cn.json', 'w'), indent=2, ensure_ascii=False)
print(open('school_cn.json').read())

{
  "课程": "PHYB57",
  "章节": 11,
  "标签": [
    "文件",
    "IO"
  ]
}


对比一下不加 `ensure_ascii=False` 的效果：

In [ ]:
# 🧪 补充实验：不加 ensure_ascii=False 会怎样？
json.dump(data_cn, open('school_ascii.json', 'w'), indent=2)
print(open('school_ascii.json').read())

{
  "\u8bfe\u7a0b": "PHYB57",
  "\u7ae0\u8282": 11,
  "\u6807\u7b7e": [
    "\u6587\u4ef6",
    "IO"
  ]
}


看到了吗 —— 中文全变成了 `\u8bfe\u7a0b` 这样的转义序列。

| 参数 | 作用 | 建议 |
|---|---|---|
| `indent=2` | 缩进 2 格，人类可读 | 配置文件必加 |
| `ensure_ascii=False` | **中文原样输出**，不转成 `\uXXXX` | 有中文就加 |

> ⚠️ 这两个参数默认都不开。**默认的 `json.dump` 输出是「挤成一行 + 中文变乱码」** —— 这是很多人第一次用 JSON 时的困惑。
>
> 💡 小知识：`\u8bfe` 这种写法**并没有错**，它是合法的 JSON，程序读回来还是「课」字。只是人看着难受而已。

## §8.2 读取 JSON 文件

In [ ]:
my_school = json.load(open('school.json', 'r'))
print(my_school)

{'school': 'UC Berkeley', 'address': {'city': 'Berkeley', 'state': 'California', 'postal': '94720'}, 'list': ['student 1', 'student 2', 'student 3']}


**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `my_school = json.load(open('school.json', 'r'))` | 打开文件、解析 JSON，得到 Python 对象。因为文件最外层是 `{ }`，所以读回来是**字典** |
| 2 | `print(my_school)` | 打印出来。注意打印出来的样子和 Python 字典**一模一样**（单引号、`True` 的写法都变回 Python 风格了）—— 因为 **JSON 已经被翻译成 Python 对象了**，不再是文本 |

> ⚠️ 课件原文说「The loaded object is a dictionary-like structure」—— 说「dictionary-**like**」是严谨的：JSON 顶层也可以是 `[ ]`，那读回来就是 `list`。**不一定是字典**，得看文件内容。

🧪 像剥洋葱一样，一层层把嵌套的值取出来：

In [ ]:
# 🧪 补充实验：怎么访问嵌套结构 —— 一层层用方括号剥开
print('第 1 层  my_school["school"]              →', my_school["school"])
print('第 2 层  my_school["address"]["city"]    →', my_school["address"]["city"])
print('第 2 层  my_school["address"]["postal"]  →', my_school["address"]["postal"], '（注意是字符串！）')
print('列表     my_school["list"][0]            →', my_school["list"][0])

第 1 层  my_school["school"]              → UC Berkeley
第 2 层  my_school["address"]["city"]    → Berkeley
第 2 层  my_school["address"]["postal"]  → 94720 （注意是字符串！）
列表     my_school["list"][0]            → student 1


> 💡 **嵌套的规律**：每往下一层，就多一个方括号。
>
> - 字典套字典 → `d["a"]["b"]`
> - 字典套列表 → `d["list"][0]`（先按**键**取，再按**下标**取）
>
> ⚠️ 注意 `"postal"` 读出来是**字符串** `'94720'` 而不是数字 94720 —— 因为当初写进去的时候带了引号。**JSON 会忠实保存你给的类型**，邮编这种前导零有意义的数据，本来就该存成字符串。

---

🧪 **JSON 的一个「失真」现象** —— 这是选格式时最该知道的一件事：

In [ ]:
# 🧪 补充实验：同一份数据，过一趟 JSON vs 过一趟 Pickle
import json, pickle

orig = {1: (10, 20)}     # 键是整数 1，值是元组

back_json = json.loads(json.dumps(orig))
back_pkl  = pickle.loads(pickle.dumps(orig))

print('原  始   ：', orig)
print('过 JSON  ：', back_json, '  ← 键变成了字符串 "1"，元组变成了列表')
print('过 Pickle：', back_pkl,  '  ← 原封不动')
print()
print('键的类型   JSON:', type(list(back_json)[0]).__name__, '  Pickle:', type(list(back_pkl)[0]).__name__)
print('值的类型   JSON:', type(back_json["1"]).__name__,   '        Pickle:', type(back_pkl[1]).__name__)

原  始   ： {1: (10, 20)}
过 JSON  ： {'1': [10, 20]}   ← 键变成了字符串 "1"，元组变成了列表
过 Pickle： {1: (10, 20)}   ← 原封不动

键的类型   JSON: str   Pickle: int
值的类型   JSON: list         Pickle: tuple


这一格信息量很大，值得慢慢看：

| | 原始 | 过一趟 JSON | 过一趟 Pickle |
|---|---|---|---|
| 键 `1` | `int` | `'1'`（**字符串**） | `1`（整数） |
| 值 `(10, 20)` | `tuple` | `[10, 20]`（**列表**） | `(10, 20)`（元组） |

> ⚠️ **为什么 JSON 会变？** 因为 JSON 的「键」规定只能是字符串，而且它**根本没有「元组」这种类型**（只有数组）。所以：
>
> - 整数键 `1` → 被转成字符串 `"1"`
> - 元组 `(10, 20)` → 被转成列表 `[10, 20]`
>
> **Pickle 则原样保留**，因为它记录的是 Python 的内部结构。
>
> 💡 **实践建议**：如果数据里有元组、有非字符串的键、有自定义对象 —— 用 Pickle（但要考虑 §7 的安全和兼容问题）。如果只是普通的字典 / 列表 / 字符串 / 数字，用 JSON 更通用。

---

# §9 · HDF5：为大数据而生的层级格式

## 课件原文（P12）

**HDF5 Files in Python**

**What is HDF5?**

- HDF5 (**Hierarchical Data Format**) is a powerful, open-source binary file format designed for handling large amounts of data. It allows fast reading/writing and supports parallel I/O.
- HDF5 files store two main types of objects:
  - **Datasets:** Array-like collections of data (similar to NumPy arrays).
  - **Groups:** Folder-like structures that can contain datasets or other groups.
- HDF5 supports **attributes** to describe properties of datasets and groups, creating a hierarchical organization similar to a filesystem.

**Why use HDF5?**

- No size limit for data storage.
- Efficient storage and retrieval through built-in optimizations.
- Commonly used in scientific computing for managing large datasets.

**Python Support for HDF5:**

- Libraries: `h5py` and `PyTables`.
- `h5py` allows Python users to create, read, and manipulate HDF5 files with ease.

## 人话版

前面四种格式都有各自的短板：

| 格式 | 短板 |
|---|---|
| TXT / CSV | 存不了几十 GB 的数据，也没法给数据挂「元数据」 |
| Pickle | 只能 Python 读，且不安全 |
| JSON | 只有基本类型，存不了大数组 |

HDF5 的解法是：**给数据加上「文件夹结构」**。

```
station.hdf5
│
├── acc/                  ← Group（组）= 文件夹
│   ├── 1/
│   │   └── data          ← Dataset（数据集）= 数组
│   │       ├── dt         = 0.04          ← Attribute（属性）= 贴在数据上的标签
│   │       ├── location   = 'Berkeley'
│   │       └── start_time = 1542000276
│   └── 2/
│       └── data
│
└── gps/
    └── 1/
        └── data
```

### 三个核心概念（记住这张表就够了）

| 概念 | 英文 | 类比 | 在代码里长什么样 |
|---|---|---|---|
| 数据集 | **Dataset** | 一个数组 / 一张表 | `hf['/acc/1/data']` |
| 组 | **Group** | 一个文件夹 | `hf['/acc/1']` |
| 属性 | **Attribute** | 贴在数据上的标签（元数据） | `hf['/acc/1/data'].attrs['dt']` |

> 💡 **为什么科学家爱用 HDF5？** 因为它把「数据」和「数据的说明」放在**同一个文件**里。三年后你打开这个文件，能直接看到「这个数组是 Berkeley 那台仪器采的、采样间隔 0.04 秒、起始时间是 2018-11-12」—— 而如果只存一个 CSV，这些信息早就不知道丢哪去了。

⚠️ `h5py` **不是标准库**，要 `pip install h5py`。

## §9.1 实操：把地震监测数据存成 HDF5

## 课件原文（P13）

**Practical Example – Storing Data in HDF5 Format**

**Scenario:** Monitoring accelerations and GPS locations in the Bay Area.

**Data:**

- Two accelerometers (Berkeley and Oakland) with different sampling rates.
- One GPS station in San Francisco sampling every 60 seconds.

**Steps for Creating HDF5 File:**

- Generate random data using NumPy for sensors in different locations.
- Create an HDF5 file (`station.hdf5`), and organize the data in groups and subgroups (`/acc/1/data`, `gps/1/data`).
- Use attributes to store additional metadata (e.g., location, start time, sampling interval).

**Sample Code:**

In [ ]:
import numpy as np
import h5py

# Data generation
acc_1 = np.random.random(1000)
location_1 = 'Berkeley'
start_time_1 = 1542000276
dt_1 = 0.04

hf = h5py.File('station.hdf5', 'w')
hf['/acc/1/data'] = acc_1
hf['/acc/1/data'].attrs['dt'] = dt_1
hf['/acc/1/data'].attrs['start_time'] = start_time_1
hf['/acc/1/data'].attrs['location'] = location_1
hf.close()

**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `import numpy as np` | 导入 numpy |
| 2 | `import h5py` | 导入 h5py —— 操作 HDF5 文件的库 |
| 4 | `# Data generation` | 注释：下面开始「造」数据（真实场景里这里应该是读仪器数据） |
| 5 | `acc_1 = np.random.random(1000)` | 造 1000 个随机数，代表 Berkeley 那台加速度计连续采到的 1000 个读数 |
| 6 | `location_1 = 'Berkeley'` | 这台仪器的位置 |
| 7 | `start_time_1 = 1542000276` | 起始时间，是 **Unix 时间戳**（从 1970-01-01 00:00:00 UTC 起算的**秒数**）。这个数对应 **2018-11-12** |
| 8 | `dt_1 = 0.04` | 采样间隔 0.04 秒 → 也就是 **25 Hz** 的采样率 |
| 10 | `hf = h5py.File('station.hdf5', 'w')` | 新建（或覆盖）一个 HDF5 文件。和 `open()` 一样，`'w'` 意味着**清空已有内容** |
| 11 | `hf['/acc/1/data'] = acc_1` | ⭐ **这一行是整节课的重点**：把数组写进路径 `/acc/1/data`。中间的 `acc` 和 `1` 两层「文件夹」h5py **会自动帮你建好**，你不用先 create_group |
| 12 | `hf['/acc/1/data'].attrs['dt'] = dt_1` | 给这个数据集挂一个**属性**：采样间隔 |
| 13 | `hf['/acc/1/data'].attrs['start_time'] = start_time_1` | 再挂一个属性：起始时间 |
| 14 | `hf['/acc/1/data'].attrs['location'] = location_1` | 再挂一个属性：位置 |
| 15 | `hf.close()` | 关闭文件 —— **数据是这一刻才真正全部落盘的** |

> ⚠️ 三个要注意的地方：
>
> 1. **路径里的 `1` 是字符串键，不是数字下标。** `'/acc/1/data'` 是纯文本，`1` 只是这一层的名字。写成 `'/acc[1]/data'` 是错的。
> 2. **`'w'` 模式会清空同名文件。** 和 §1 的坑一模一样。
> 3. **忘了 `close()` 的话，文件可能是不完整的。** 规范写法是 `with h5py.File(...) as hf:`。

🧪 眼见为实 —— 看看 h5py 到底帮我们建出了什么结构：

In [ ]:
# 🧪 补充实验：把 HDF5 文件的内部结构整个打印出来
with h5py.File('station.hdf5', 'r') as f:
    def show(name, obj):
        kind = 'Group  ' if isinstance(obj, h5py.Group) else 'Dataset'
        attrs = dict(obj.attrs)
        print(f'{kind} /{name}', ('   ← ' + str(attrs)) if attrs else '')
    f.visititems(show)

Group   /acc 
Group   /acc/1 
Dataset /acc/1/data    ← {'dt': np.float64(0.04), 'location': 'Berkeley', 'start_time': np.int64(1542000276)}


结构一目了然：

- `Group /acc` 和 `Group /acc/1` 是 h5py **自动创建**的两层组（就是那个「一行写三层」的效果）
- `Dataset /acc/1/data` 是真正存数组的地方
- 后面跟着的就是三个**属性** —— 它们和数据**待在同一个文件里**

🧪 课件描述的场景里其实有**两台**加速度计（Berkeley 和 Oakland）和**一台** GPS，但示例代码只写了 Berkeley 那一台。补全它：

In [ ]:
# 🧪 补充实验：把课件描述里的完整场景补全（2 台加速度计 + 1 台 GPS）
with h5py.File('station_full.hdf5', 'w') as hf:
    # 两台加速度计：采样率不同
    for i, (loc, dt) in enumerate([('Berkeley', 0.04), ('Oakland', 0.02)], start=1):
        g = hf.create_group(f'/acc/{i}')
        g.create_dataset('data', data=np.random.random(1000))
        g['data'].attrs['location'] = loc
        g['data'].attrs['dt'] = dt
        g['data'].attrs['start_time'] = 1542000276

    # 一台 GPS：每 60 秒一个点，(纬度, 经度)
    g = hf.create_group('/gps/1')
    g.create_dataset('data', data=np.random.random((100, 2)))
    g['data'].attrs['location'] = 'San Francisco'
    g['data'].attrs['dt'] = 60
    g['data'].attrs['start_time'] = 1542000276

print('文件建好了。里面的结构：')
with h5py.File('station_full.hdf5', 'r') as f:
    def show(name, obj):
        kind = 'Group  ' if isinstance(obj, h5py.Group) else 'Dataset'
        attrs = dict(obj.attrs)
        print(f'{kind} /{name}', ('   ← ' + str(attrs)) if attrs else '')
    f.visititems(show)

文件建好了。里面的结构：
Group   /acc 
Group   /acc/1 
Dataset /acc/1/data    ← {'dt': np.float64(0.04), 'location': 'Berkeley', 'start_time': np.int64(1542000276)}
Group   /acc/2 
Dataset /acc/2/data    ← {'dt': np.float64(0.02), 'location': 'Oakland', 'start_time': np.int64(1542000276)}
Group   /gps 
Group   /gps/1 
Dataset /gps/1/data    ← {'dt': np.int64(60), 'location': 'San Francisco', 'start_time': np.int64(1542000276)}


> 💡 对比一下：要表达「同一个文件里放多台仪器的数据」，用 CSV 你得建 3 个文件再加一个 README 说明；用 HDF5 **一个文件**就装下了，而且结构自带说明。

## §9.2 读取 HDF5 文件

## 课件原文（P14）

**Reading HDF5 Files**

**How to Read Data from HDF5:**

- Open the file in read mode (`'r'`).
- Use the `keys()` function to explore the structure of the file (e.g., see the groups `acc` and `gps`).
- Access datasets directly using paths (e.g., `hf_in['/acc/1/data']`).

**Example:**

In [ ]:
hf_in = h5py.File('station.hdf5', 'r')
acc_data = hf_in['/acc/1/data'][:10]  # Access first 10 entries
acc_attrs = hf_in['/acc/1/data'].attrs['location']  # Access metadata
print(acc_data, acc_attrs)  # Output data and location

[0.55399874 0.9582336  0.31390566 0.25641815 0.10927906 0.69812411
 0.12189449 0.34568028 0.44407176 0.38451   ] Berkeley


**📖 这一格在做什么**

| 行 | 代码 | 人话 |
|---|---|---|
| 1 | `hf_in = h5py.File('station.hdf5', 'r')` | 以**只读**模式打开（`'r'`），表示只看不改 |
| 2 | `acc_data = hf_in['/acc/1/data'][:10]` | 按路径取出数据集，`[:10]` 取**前 10 个**读数。后面的注释 `# Access first 10 entries` 就是「访问前 10 条」 |
| 3 | `acc_attrs = hf_in['/acc/1/data'].attrs['location']` | 取出挂在数据上的**属性** `location`。`.attrs` 就是「属性字典」 |
| 4 | `print(acc_data, acc_attrs)` | 把数据和位置一起打印出来 |

> ⚠️ **两个坑：**
>
> 1. **`hf_in` 从头到尾没有被 close！** 课件忘了写 `hf_in.close()`。Jupyter 里影响不大，但在长期运行的程序里会**泄漏文件句柄**。规范写法是 `with h5py.File(...) as hf_in:`。
> 2. **`[:10]` 这个切片不能省。** 这不是可选的美化 —— 见下面的实验。

🧪 **HDF5 是「懒」的** —— 这是一个非常重要的特性：

In [ ]:
# 🧪 补充实验：不加切片，拿到的是「数据集对象」，不是数组
ds = hf_in['/acc/1/data']
print('不加切片 →', type(ds))
print('          形状', ds.shape, ' 类型', ds.dtype, ' （这些信息不用读数据就知道）')
print('          前 3 个值', ds[:3])
print()
arr = hf_in['/acc/1/data'][:]
print('加了切片 →', type(arr))
print('          形状', arr.shape, ' ← 数据这才真正从磁盘读进内存')

不加切片 → <class 'h5py._hl.dataset.Dataset'>
          形状 (1000,)  类型 float64  （这些信息不用读数据就知道）
          前 3 个值 [0.55399874 0.9582336  0.31390566]

加了切片 → <class 'numpy.ndarray'>
          形状 (1000,)  ← 数据这才真正从磁盘读进内存


> 💡 **为什么要这样设计？** 因为 HDF5 里的数组可能有几十 GB。如果 `hf['/data']` 直接返回整个数组，内存立刻就爆了。
>
> 所以 h5py 让你先拿到一个**「数据集对象」**（一个指向磁盘数据的「遥控器」），只有当你**切片**的时候，才真正把那一部分读进内存。
>
> | 写法 | 拿到什么 | 会不会读盘 |
> |---|---|---|
> | `hf['/acc/1/data']` | Dataset 对象（遥控器） | ❌ 不读 |
> | `hf['/acc/1/data'][:]` | 整个数组 | ✅ 全读 |
> | `hf['/acc/1/data'][:10]` | 前 10 个 | ✅ 只读 10 个 |
>
> 这就是课件说的「Efficient storage and retrieval」的真正含义：**想读多少读多少**。

🧪 课件提到 `keys()` 但代码里没写，补上 —— 这是「探路」最常用的方法：

In [ ]:
# 🧪 补充实验：keys() 逐层探路，不用打开文件也知道里面有什么
print('顶层有哪些组？      ', list(hf_in.keys()))
print('acc 下面有什么？    ', list(hf_in['acc'].keys()))
print('acc/1 下面有什么？  ', list(hf_in['/acc/1'].keys()))
print()
print('这个数据集挂了哪些属性？', list(hf_in['/acc/1/data'].attrs))
print('逐个打出来：')
for k in hf_in['/acc/1/data'].attrs:
    print(f'   {k:12s} = {hf_in["/acc/1/data"].attrs[k]}')
hf_in.close()   # 课件忘了关，我们补上
print()
print('文件已关闭。')

顶层有哪些组？       ['acc']
acc 下面有什么？     ['1']
acc/1 下面有什么？   ['data']

这个数据集挂了哪些属性？ ['dt', 'location', 'start_time']
逐个打出来：


   dt           = 0.04
   location     = Berkeley
   start_time   = 1542000276

文件已关闭。


掌握这四个动作，你就能自由探索任何 HDF5 文件了：

| 想知道什么 | 用什么 |
|---|---|
| 里面有哪几组？ | `list(hf.keys())` |
| 这一组下面有什么？ | `list(hf['组名'].keys())` |
| 整个文件的结构？ | `hf.visititems(函数)` |
| 数据挂了哪些元数据？ | `list(hf['路径'].attrs)` |

> 💡 拿到一个陌生的 `.hdf5` 文件时，第一件事就是 `visititems` 打印整个结构 —— 比一个一个猜路径快得多。

---

# §10 · 收尾：五种格式到底怎么选

## 一句话版本

| 你想干的事 | 用哪个 |
|---|---|
| 存一段文字、一个日志 | **TXT** |
| 存一张能塞进 Excel 的表格 | **CSV** |
| 存 Python 对象（字典、自定义类），只给自己用 | **Pickle** |
| 存配置 / 和别的语言交换数据 / 给 API 用 | **JSON** |
| 存几百 MB 以上的科学数据，还要带元数据 | **HDF5** |

## 完整对照表

| | TXT | CSV | Pickle | JSON | HDF5 |
|---|---|---|---|---|---|
| **类型** | 纯文本 | 纯文本 | 二进制 | 纯文本 | 二进制 |
| **人能直接读吗** | ✅ | ✅ | ❌ 乱码 | ✅ | ❌ 需要工具 |
| **跨语言** | ✅ | ✅ | ❌ 只 Python | ✅ | ✅ 有 C/Fortran/MATLAB 库 |
| **能存任意 Python 对象** | ❌ | ❌ | ✅ | ❌ | ❌（但能存任意数组） |
| **能存大数据** | ❌ | ❌ | 一般 | ❌ | ✅ **没有大小限制** |
| **支持元数据** | 靠自己约定 | ❌ | ✅（对象本身） | 靠嵌套 | ✅ **Attribute** |
| **安全** | ✅ | ✅ | ⚠️ **会执行代码** | ✅ | ✅ |
| **适合长期归档** | ✅ | ✅ | ❌ | ✅ | ✅ |

---

🧪 最后做个实测 —— **同一份数据，五种格式比大小和速度**（顺便验证 §8 那个勘误）：

In [ ]:
# 🧪 补充实验：同一份数据，五种格式正面对比
import os, time, json, pickle
import numpy as np, h5py

data = np.random.random((10000, 5))      # 5 万个数字，约 400 KB 的原始数据
results = []

t = time.perf_counter(); json.dump(data.tolist(), open('cmp.json', 'w'))
results.append(('JSON',   'cmp.json',   time.perf_counter() - t))

t = time.perf_counter(); pickle.dump(data, open('cmp.pkl', 'wb'))
results.append(('Pickle', 'cmp.pkl',    time.perf_counter() - t))

t = time.perf_counter(); np.savetxt('cmp.csv', data, fmt='%.6f', delimiter=',')
results.append(('CSV',    'cmp.csv',    time.perf_counter() - t))

t = time.perf_counter(); np.save('cmp.npy', data)
results.append(('npy',    'cmp.npy',    time.perf_counter() - t))

t = time.perf_counter()
with h5py.File('cmp.h5', 'w') as h:
    h['data'] = data
results.append(('HDF5',   'cmp.h5',     time.perf_counter() - t))

print(f"{'格式':<8}{'文件大小':>12}{'写入耗时':>12}")
print('-' * 34)
for name, fn, dt in sorted(results, key=lambda r: os.path.getsize(r[1])):
    print(f'{name:<8}{os.path.getsize(fn)/1024:>9.1f} KB{dt*1000:>9.1f} ms')

格式              文件大小        写入耗时
----------------------------------
npy         390.8 KB      0.6 ms
Pickle      390.8 KB      0.3 ms
HDF5        392.6 KB      1.1 ms
CSV         439.5 KB     13.8 ms
JSON       1009.1 KB     34.3 ms


**结论（每次运行数字会略有不同，但量级是稳定的）：**

1. **📌 课件 P10 的勘误被证实了**：JSON 不但不比 Pickle 小，反而**大了 2 倍多**；写入速度也**慢了几十倍**。原因是 JSON 把所有数字都变成了**文本**（`0.123456` 要 8 个字符），而 Pickle 直接存二进制（8 个字节）。
2. **文本格式（CSV / JSON）都比二进制格式大**，因为它们把数字当文字存。
3. **npy / Pickle / HDF5 体积接近** —— 它们存的都是原始二进制。
4. **HDF5 的速度和体积与 npy 相当**，但它额外支持**层级结构 + 元数据 + 部分读取**，这是 npy 做不到的。

> 💡 **所以选格式的正确思路不是「哪个更小更快」，而是「哪个更合适」**：
>
> - 要给 Excel 看 → CSV（哪怕它大一点）
> - 要跨语言、要安全 → JSON
> - 要存超大科学数据 + 元数据 → HDF5
> - 只是自己程序内部临时缓存一下 → Pickle 最快最省事，但**别 load 陌生文件**

---

🧪 最后看一眼这个 notebook 一共在磁盘上留下了哪些文件：

In [ ]:
# 🧪 本 notebook 生成的所有数据文件
import os
total = 0
for fn in sorted(os.listdir('.')):
    if os.path.isfile(fn):
        size = os.path.getsize(fn)
        total += size
        print(f'{size:>9,} B   {fn}')
print('-' * 30)
print(f'{total:>9,} B   共 {len([f for f in os.listdir(".") if os.path.isfile(f)])} 个文件')
print()
print('当前目录：', os.getcwd())

  450,000 B   cmp.csv
  402,048 B   cmp.h5
1,033,279 B   cmp.json
  400,128 B   cmp.npy
  400,163 B   cmp.pkl
       44 B   mini.csv
      272 B   mixed.pkl
       47 B   my_arr.txt
      149 B   school.json
      106 B   school_ascii.json
       82 B   school_cn.json
   16,192 B   station.hdf5
   29,256 B   station_full.hdf5
    2,521 B   test.csv
       34 B   test.pkl
       96 B   test.txt
       34 B   test2.pkl
------------------------------
2,734,451 B   共 17 个文件

当前目录： /private/tmp/phyb57/exec/phyb57_c11_data


---

# ✅ 学完了，回头看一眼

这一章你实际掌握了 **5 种格式 × 读写双向 = 10 个操作**：

| 格式 | 写（存） | 读（取） |
|---|---|---|
| **TXT** | `open(..., 'w')` / `'a'` → `write()` | `open(..., 'r')` → `read()` / `readlines()` |
| **CSV（numpy）** | `np.savetxt(..., delimiter=',')` | `np.loadtxt(..., delimiter=',')` |
| **CSV（pandas）** | `df.to_csv()` | `pd.read_csv()` |
| **Pickle** | `pickle.dump(obj, f)` | `pickle.load(f)` |
| **JSON** | `json.dump(obj, f)` | `json.load(f)` |
| **HDF5** | `hf['/path'] = arr` + `.attrs[...]` | `hf['/path'][:]` + `.attrs[...]` |

## 三个最值钱的坑（考试和实际工作都会遇到）

1. ⚠️ **`'w'` 会清空文件** —— 它是「先擦干净再写」，不是「写入」。想追加用 `'a'`。
2. ⚠️ **读 CSV 必须两边说好分隔符** —— `np.loadtxt` 少写 `delimiter=','` 就报 `could not convert string to float`。
3. ⚠️ **不要 `pickle.load` 陌生文件** —— 它会执行文件里的代码，这是真实的攻击面。

## 三个最值钱的好习惯

1. ✅ **一律用 `with open(...) as f:`** —— 自动关闭，报错也关，不用记 `close()`。
2. ✅ **`json.dump` 记得加 `indent=2` 和 `ensure_ascii=False`** —— 否则文件是「挤成一行 + 中文乱码」。
3. ✅ **浮点数比较用 `np.allclose()`，不用 `==`** —— 因为浮点是近似存储的。

> 课件 14 页、16 段代码，到这里全部覆盖并逐行讲完了。